# NeuroSleep — Sleep Stage Classification

**Improved Student — 99,477 parameters, 93% accuracy**

| Metric | Value |
|--------|-------|
| Accuracy | 93.0% ± 1.0% |
| Cohen's κ | 0.861 ± 0.027 |
| Macro F1 | 0.794 ± 0.036 |

Classifies 30-second PSG epochs into Wake, N1, N2, N3, REM.

- **GitHub:** https://github.com/shamiquekhan/neuromorphic-sleep-staging-pipeline
- **HF Model:** https://huggingface.co/shamique/Light-Weight-Neuromorphic-Sleep-Stage-Model
- **Demo:** https://huggingface.co/spaces/shamique/neurosleep-demo

## 01 — Install & Environment

In [ ]:
!pip install -q torch numpy safetensors huggingface_hub

import torch
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 02 — Model Definition (self-contained)

This cell embeds the full `ImprovedStudent` architecture so the notebook runs without cloning the repo.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass(frozen=True)
class StudentConfig:
    n_channels: int = 4
    n_classes: int = 5
    sampling_rate: int = 100
    gru_hidden: int = 64
    gru_layers: int = 2
    stem_width: int = 10
    encoder_channels: tuple = (32, 32)
    gabor_n_filters: int = 8
    gabor_out_dim: int = 32
    seq_len: int = 10
    epoch_seconds: int = 30

    @property
    def samples_per_epoch(self) -> int:
        return self.epoch_seconds * self.sampling_rate

    @property
    def input_shape(self) -> tuple:
        return (1, self.seq_len, self.n_channels, self.samples_per_epoch)

    @property
    def output_shape(self) -> tuple:
        return (1, self.seq_len, self.n_classes)


class ImprovedStudent(nn.Module):
    """99,477-parameter sleep-stage classifier.
    
    Multi-Res Stem -> Depthwise-Separable CNN -> Gabor FEB -> 2-layer GRU
    """

    def __init__(self, config: StudentConfig | None = None):
        super().__init__()
        if config is None:
            config = StudentConfig()
        self.config = config

        self.stem_s = nn.Sequential(
            nn.Conv1d(4, 8, 25, stride=6, bias=False),
            nn.BatchNorm1d(8),
        )
        self.stem_l = nn.Sequential(
            nn.Conv1d(4, 8, 200, stride=25, bias=False),
            nn.BatchNorm1d(8),
        )

        self.enc = nn.ModuleDict({
            "0": nn.ModuleDict({
                "dw": nn.Conv1d(16, 16, 5, stride=2, padding=2, groups=16, bias=False),
                "pw": nn.Conv1d(16, 32, 1, bias=False),
                "bn": nn.BatchNorm1d(32),
            }),
            "1": nn.ModuleDict({
                "dw": nn.Conv1d(32, 32, 5, stride=2, padding=2, groups=32, bias=False),
                "pw": nn.Conv1d(32, 32, 1, bias=False),
                "bn": nn.BatchNorm1d(32),
            }),
        })

        self.pool = nn.AdaptiveAvgPool1d(8)

        self.gabor_freq = nn.Parameter(
            torch.linspace(0.5, 30.0, config.gabor_n_filters) / config.sampling_rate
        )
        self.gabor_sigma = nn.Parameter(
            torch.full((config.gabor_n_filters,), 0.02)
        )
        self.gab_proj = nn.Linear(config.gabor_n_filters, 16)

        feature_dim = 32 * 8 + 16  # 272
        self.gru = nn.GRU(
            feature_dim, config.gru_hidden,
            num_layers=config.gru_layers, batch_first=True,
        )
        self.head = nn.Linear(config.gru_hidden, config.n_classes)
        self.feature_dim = feature_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t, c, s = x.shape
        flat = x.reshape(b * t, c, s)

        short = F.relu6(self.stem_s[1](self.stem_s[0](flat)))
        long = F.relu6(self.stem_l[1](self.stem_l[0](flat)))
        target = min(short.shape[-1], long.shape[-1])
        short = F.adaptive_avg_pool1d(short, target)
        long = F.adaptive_avg_pool1d(long, target)
        stem_out = torch.cat([short, long], dim=1)

        e0 = F.relu6(self.enc["0"]["bn"](self.enc["0"]["pw"](self.enc["0"]["dw"](stem_out))))
        e1 = F.relu6(self.enc["1"]["bn"](self.enc["1"]["pw"](self.enc["1"]["dw"](e0))))

        cnn = self.pool(e1).flatten(1)

        kernel_size = 51
        t_axis = torch.arange(
            -(kernel_size // 2), kernel_size // 2 + 1,
            dtype=torch.float32, device=x.device,
        ).unsqueeze(0)
        f0 = self.gabor_freq.unsqueeze(1)
        sigma = (self.gabor_sigma.abs() + 1e-4).unsqueeze(1)
        envelope = torch.exp(-0.5 * (t_axis / (sigma * kernel_size)) ** 2)
        carrier = torch.cos(2 * 3.14159265 * f0 * t_axis)
        kernels = (envelope * carrier).unsqueeze(1)
        x_mean = flat.mean(dim=1, keepdim=True)
        gab = F.conv1d(x_mean, kernels, padding=kernel_size // 2)
        gab = F.adaptive_avg_pool1d(gab, 1).squeeze(-1)
        gab = self.gab_proj(gab)

        features = torch.cat([cnn, gab], dim=-1).reshape(b, t, self.feature_dim)
        sequence_out, _ = self.gru(features)
        logits = self.head(sequence_out)
        return logits


STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
config = StudentConfig()
print(f'Model defined: {sum(p.numel() for p in ImprovedStudent().parameters()):,} parameters')

## 03 — Load Weights from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

ckpt_path = hf_hub_download(
    repo_id='shamique/Light-Weight-Neuromorphic-Sleep-Stage-Model',
    filename='student_full_finetuned.safetensors',
)

model = ImprovedStudent(config)
model.load_state_dict(load_file(ckpt_path, device='cpu'))
model.eval()

print(f'Model loaded from {ckpt_path}')

## 04 — Run Inference

In [ ]:
import time

# Create synthetic demo input: [batch=1, 10 epochs, 4 channels, 3000 samples]
# Replace this with real preprocessed PSG data
x = torch.randn(1, 10, 4, 3000)

start = time.perf_counter()
with torch.inference_mode():
    logits = model(x)           # [1, 10, 5]
    probs = torch.softmax(logits, dim=-1)
    preds = probs.argmax(dim=-1)  # [1, 10]
latency_ms = (time.perf_counter() - start) * 1000

print(f'Input shape:  {list(x.shape)}')
print(f'Output shape: {list(logits.shape)}')
print(f'Latency:      {latency_ms:.1f} ms (CPU)')
print()

for i in range(10):
    stage = STAGE_NAMES[preds[0, i].item()]
    conf = probs[0, i, preds[0, i]].item()
    print(f'  Epoch {i:2d}: {stage:>4} ({conf:.1%})')

## 05 — Per-Class Results

In [ ]:
per_class = {
    'Wake': {'f1': 0.982, 'precision': 0.997, 'recall': 0.967},
    'N1':   {'f1': 0.464, 'precision': 0.400, 'recall': 0.552},
    'N2':   {'f1': 0.886, 'precision': 0.928, 'recall': 0.848},
    'N3':   {'f1': 0.877, 'precision': 0.845, 'recall': 0.912},
    'REM':  {'f1': 0.808, 'precision': 0.714, 'recall': 0.930},
}

print('Per-Class Performance (4-fold CV, 15 subjects)')
print('=' * 55)
print(f'{"Stage":>6} {"F1":>8} {"Precision":>10} {"Recall":>8}')
print('-' * 55)
for stage, v in per_class.items():
    print(f'{stage:>6} {v["f1"]:>8.3f} {v["precision"]:>10.3f} {v["recall"]:>8.3f}')

## 06 — Confusion Matrix

In [ ]:
stages = ['Wake', 'N1', 'N2', 'N3', 'REM']
cm = [
    [0.967, 0.000, 0.033, 0.000, 0.000],
    [0.000, 0.552, 0.448, 0.000, 0.000],
    [0.000, 0.020, 0.848, 0.080, 0.052],
    [0.000, 0.000, 0.088, 0.912, 0.000],
    [0.000, 0.000, 0.070, 0.000, 0.930],
]

print('Confusion Matrix (row-normalized)')
print('=' * 50)
print(f'{"":>6}', end='')
for s in stages:
    print(f'{s:>8}', end='')
print()
for i, row in enumerate(cm):
    print(f'{stages[i]:>6}', end='')
    for val in row:
        print(f'{val:>8.3f}', end='')
    print()

## 07 — Model Efficiency

In [ ]:
n_params = sum(p.numel() for p in model.parameters())

print('Model Efficiency')
print('=' * 40)
print(f'  Parameters:     {n_params:,}')
print(f'  Model size:     ~{n_params * 4 / 1024:.0f} KB (FP32)')
print(f'  Input shape:    {list(config.input_shape)}')
print(f'  Output shape:   {list(config.output_shape)}')
print(f'  Context:        {config.seq_len * config.epoch_seconds} seconds')
print(f'  Latency (CPU):  {latency_ms:.1f} ms')

## 08 — LoRA Adaptation (Parameter-Efficient Fine-Tuning)

Apply **LoRA** to fine-tune on your own sleep dataset with only **552 trainable parameters** (0.55% of the model).

In [ ]:
!pip install -q peft

from peft import LoraConfig, get_peft_model, TaskType

# Load fresh model
lora_model = ImprovedStudent(config)
lora_model.load_state_dict(load_file(ckpt_path, device='cpu'))

# Apply LoRA to classification head
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    target_modules=['head'],
    lora_dropout=0.05,
    bias='none',
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
# Verify LoRA model works
with torch.inference_mode():
    lora_logits = lora_model(x)

print(f'LoRA output shape: {list(lora_logits.shape)}')
print(f'Output matches base: {torch.allclose(logits, lora_logits, atol=1e-5)}')
print()
print('LoRA vs Full Fine-Tuning:')
print('  LoRA r=8:        552 params (0.55%) -> ~89% accuracy')
print('  Full Fine-Tune:  99,477 params      -> 93% accuracy')

## 09 — Links

- **GitHub:** https://github.com/shamiquekhan/neuromorphic-sleep-staging-pipeline
- **HF Model:** https://huggingface.co/shamique/Light-Weight-Neuromorphic-Sleep-Stage-Model
- **HF Demo:** https://huggingface.co/spaces/shamique/neurosleep-demo